In [1]:
import sys
import os
import glob

import re

import itertools

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
sys.path.append("../model/")
import utrdata_cl as utrdata
from legnet import LegNetClassifier
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr5"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
PATH_FROM = f"./{utr_type.upper()}_zscores_2024-06-04.csv"
df = pd.read_csv(PATH_FROM)

In [6]:
assert (df["seq"] == df["seq"].str.upper()).all()

In [7]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

5

In [8]:
batch_size = 1024
num_workers = 32

In [9]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

model = RNARegressor.load_from_checkpoint(MODEL_PATH)
trainer = pl.Trainer(
    callbacks=[pl.callbacks.TQDMProgressBar(refresh_rate=0.5)],
    logger=False,
    accelerator="gpu",
    devices=[0],
    deterministic=True,
    num_sanity_val_steps=0,
)
prediction = trainer.predict(model=model, dataloaders=dl_test)
test_pred, test_real = zip(*prediction)
test_pred = torch.concat(test_pred).numpy()
test_real = torch.concat(test_real).numpy()
df_pred = df.copy()
df_pred["pred_delta"] = test_pred[:, 0]
df_pred["pred_mass_center"] = test_pred[:, 1]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [10]:
df_pred

,seq,cell_type,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std,pred_delta,pred_mass_center
0,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c1,614.0,395.0,685.0,815.0,2.677959,2.296948,0.381012,1.191578,0.319754,0.020646,2.235352
1,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c17,3442.0,3260.0,2909.0,4201.0,2.569722,2.296948,0.272774,0.853076,0.319754,-0.073548,2.374466
2,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c2,3384.0,2467.0,1713.0,1320.0,2.109072,2.296948,-0.187875,-0.587562,0.319754,0.003716,2.172375
3,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c4,3926.0,1645.0,1375.0,933.0,1.913060,2.296948,-0.383888,-1.200572,0.319754,-0.008880,2.328300
4,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c6,4475.0,2894.0,3129.0,2018.0,2.214925,2.296948,-0.082023,-0.256519,0.319754,-0.019559,2.336071
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50865,TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCG...,c1,3743.0,3008.0,4050.0,4807.0,2.635636,2.490269,0.145367,1.232848,0.117911,-0.036285,2.368677
50866,TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCG...,c17,8932.0,12713.0,10469.0,9848.0,2.506005,2.490269,0.015737,0.133462,0.117911,0.054916,2.555843
50867,TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCG...,c2,9045.0,10634.0,12367.0,9488.0,2.536861,2.490269,0.046593,0.395149,0.117911,-0.022615,2.420760
50868,TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCG...,c4,15230.0,14372.0,13533.0,13944.0,2.458855,2.490269,-0.031413,-0.266416,0.117911,-0.016199,2.403863


## Adding metadata

In [11]:
df_src = pd.read_table(f"../../data/lib2/raw/{utr_type.upper()}_sequence_metadata_06_04_24.tsv", index_col=0)
df.index.name = "seq"

In [12]:
def get_tag_rowwise(row: pd.Series):
    ct1 = row['first_cell_line']
    ct2 = row['second_cell_line']
    if pd.isna(ct1):
        return {"direction": np.nan, "cell_types": np.nan}
    elif pd.isna(ct2):
        if ct1 in {"up", "down"}:
            return {"direction": ct1, "cell_types": np.nan}
        else:
            raise Exception
    else:
        return {"direction": np.nan, "cell_types": f"{ct1}-{ct2}"}

In [13]:
tags_df = df_src.apply(get_tag_rowwise, axis=1).apply(pd.Series)

### Info for diffusion

In [14]:
diffusion_files = glob.glob(f"../../generator/diffusion/generate/results/*{utr_type[-1]}UTR.csv")
diffusion_taken = []
for gen_result_file in tqdm(diffusion_files):
    bn = os.path.basename(gen_result_file)
    ct = re.match(r"cell_(c\d+)_epoch_\d+_\dUTR.csv", bn).group(1)
    gen_result_df = pd.read_csv(gen_result_file)
    gen_result_df = gen_result_df[gen_result_df["seq"].isin(set(df["seq"]))]
    gen_result_df["diffusion_cell_type"] = ct
    diffusion_taken.append(gen_result_df)
diffusion_taken = pd.concat(diffusion_taken).set_index("seq")

  0%|          | 0/4 [00:00<?, ?it/s]

In [15]:
diffusion_taken = diffusion_taken[~diffusion_taken.index.duplicated(keep=False)]

## Reshaping data

In [16]:
df_sub = df_pred.set_index(["seq", "cell_type"]).copy()
df_new = df_sub[["mass_center", "diff", "pred_mass_center", "pred_delta"]]
df_new = df_new.unstack(1).reorder_levels((1, 0), axis=1).sort_index(axis=1)

In [17]:
df_new["source"] = df_src["source"]
df_new["source_ext"] = df_src["name"]
df_new["diffusion_query"] = diffusion_taken["mass_center"]
df_new["diffusion_cell_type"] = diffusion_taken["diffusion_cell_type"]
df_new["direction"] = tags_df["direction"]
df_new["cell_types_directional"] = tags_df["cell_types"]
df_new["cell_types_ordered"] = np.nan

In [18]:
cell_types = ["c1", "c2", "c4", "c6", "c13", "c17"]
for ct1, ct2 in itertools.combinations(cell_types, 2):
    df_new.loc[df_new["cell_types_directional"] == f"{ct1}-{ct2}", "cell_types_ordered"] = f"{ct1}-{ct2}"
    df_new.loc[df_new["cell_types_directional"] == f"{ct1}-{ct2}", "direction"] = "up"
    df_new.loc[df_new["cell_types_directional"] == f"{ct2}-{ct1}", "cell_types_ordered"] = f"{ct1}-{ct2}"
    df_new.loc[df_new["cell_types_directional"] == f"{ct2}-{ct1}", "direction"] = "down"

In [19]:
df_new

cell_type                                                 c1              \
                                                        diff mass_center   
seq                                                                        
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC  0.381012    2.677959   
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC -0.023875    2.591365   
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC  0.213143    2.419520   
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG  0.005109    2.676759   
AAAAAGAGGCCTGGGCCGGGAGTCCCCGCCCCTGGGCCCAGCCCTGGCCG  0.182111    2.467139   
...                                                      ...         ...   
TTTTTCGGCGGCGGGTGGGGGGGGGCGGTTTCGTTCGTGGTCCTTTGTTT -0.004011    2.637498   
TTTTTCTTCCTCCGCCCCTTCCTGGTAGAGGAGAAGGCGCGGGTGGGGGA -0.043220    2.719421   
TTTTTGCGTGTGCGCTGTGACTGGTGCTCCGTTTTCCTCGCTCATGGTGG  0.239196    2.513599   
TTTTTTCTGCTGCAGGCGGAGCGGCGCGCGGAGGAGGCGCCGGGAGCCGC  0.124972    2.735958   
TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCGTGGG  0.145367    2.635636   

cell_type                                                      \
                                                   pred_delta   
seq                                                             
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC   0.020646   
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC   0.017169   
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC   0.018511   
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG   0.032640   
AAAAAGAGGCCTGGGCCGGGAGTCCCCGCCCCTGGGCCCAGCCCTGGCCG  -0.004608   
...                                                       ...   
TTTTTCGGCGGCGGGTGGGGGGGGGCGGTTTCGTTCGTGGTCCTTTGTTT   0.001343   
TTTTTCTTCCTCCGCCCCTTCCTGGTAGAGGAGAAGGCGCGGGTGGGGGA   0.007573   
TTTTTGCGTGTGCGCTGTGACTGGTGCTCCGTTTTCCTCGCTCATGGTGG  -0.013145   
TTTTTTCTGCTGCAGGCGGAGCGGCGCGCGGAGGAGGCGCCGGGAGCCGC  -0.023647   
TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCGTGGG  -0.036285   

cell_type                                                                 c17  \
                                                   pred_mass_center      diff   
seq                                                                             
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC         2.235352  0.272774   
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC         2.307761 -0.240933   
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC         2.419005 -0.058341   
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG         2.466333  0.100583   
AAAAAGAGGCCTGGGCCGGGAGTCCCCGCCCCTGGGCCCAGCCCTGGCCG         2.158932 -0.375713   
...                                                             ...       ...   
TTTTTCGGCGGCGGGTGGGGGGGGGCGGTTTCGTTCGTGGTCCTTTGTTT         2.642587  0.216876   
TTTTTCTTCCTCCGCCCCTTCCTGGTAGAGGAGAAGGCGCGGGTGGGGGA         2.633624  0.284050   
TTTTTGCGTGTGCGCTGTGACTGGTGCTCCGTTTTCCTCGCTCATGGTGG         2.406738 -0.013046   
TTTTTTCTGCTGCAGGCGGAGCGGCGCGCGGAGGAGGCGCCGGGAGCCGC         2.684644  0.031983   
TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCGTGGG         2.368677  0.015737   

cell_type                                                                  \
                                                   mass_center pred_delta   
seq                                                                         
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC    2.569722  -0.073548   
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC    2.374306  -0.045304   
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC    2.148036  -0.051771   
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG    2.772232  -0.031547   
AAAAAGAGGCCTGGGCCGGGAGTCCCCGCCCCTGGGCCCAGCCCTGGCCG    1.909315  -0.109573   
...                                                        ...        ...   
TTTTTCGGCGGCGGGTGGGGGGGGGCGGTTTCGTTCGTGGTCCTTTGTTT    2.858386   0.079729   
TTTTTCTTCCTCCGCCCCTTCCTGGTAGAGGAGAAGGCGCGGGTGGGGGA    3.046692   0.045410   
TTTTTGCGTGTGCGCTGTGACTGGTGCTCCGTTTTCCTCGCT

In [20]:
df_new.to_parquet(f"{utr_type.upper()}_library2_preds.parquet")

---